In [ ]:
import sys
sys.path.append('../../')

from pathlib import Path

import phys_ml.visualization.vertex_visualization as vertvis
import phys_ml.visualization.base as vis
from phys_ml.analysis.vertex import *
from phys_ml.evaluation import vertex as verteval
from phys_ml.load_data.vertex import *
from phys_ml.trainer.vertex import *
from phys_ml.util import slurm_generate

data_dir = Path('/gpfs/data/fs71925/shepp123/frgs_6d/')
base_path = '/gpfs/data/fs71925/shepp123/PhysML/saves/vertex_24x6/run_results/'
rmse_df = pd.read_csv(base_path + 'reconstruction_results.csv')
classification_df = pd.read_csv(base_path + 'classification_results.csv')

## contents

&emsp;0&emsp;on real space  
&emsp;1&emsp;trained on all vertices  
&emsp;&emsp;1-1&emsp;simple autoencoder  
&emsp;&emsp;1-2&emsp;contrastive autoencoder  
&emsp;2&emsp;trained on only 2 phases  
&emsp;&emsp;2-1&emsp;exclude SC-phase  
&emsp;&emsp;&emsp;2-1-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-1-2&emsp;contrastive autoencoder  
&emsp;&emsp;2-2&emsp;exclude AFM-phase  
&emsp;&emsp;&emsp;2-2-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-2-2&emsp;contrastive autoencoder  
&emsp;&emsp;2-3&emsp;exclude FM-phase  
&emsp;&emsp;&emsp;2-3-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-3-2&emsp;contrastive autoencoder  
&emsp;3&emsp;trained on only 1 phase  
&emsp;&emsp;3-1&emsp;SC-phase  
&emsp;&emsp;3-2&emsp;AFM-phase  
&emsp;&emsp;3-3&emsp;FM-phase  
&emsp;4&emsp;correlation  
&emsp;5&emsp;convergence  

## visualize vertex

In [ ]:
files_names = ['tp0.000000_mu0.000000', 'tp0.270000_mu1.080000', 'tp0.500000_mu2.000000']
vertices = [AutoEncoderVertex24x6Dataset.load_from_file(data_dir / f'{fn}.h5') for fn in files_names]
vertvis.plot_compare_slices(vertices)

## vertex statistics

In [ ]:
verteval.vertex_statistics(data_dir)

## correlation

In [ ]:
cor_mat = np.load(f'cor_mat_vertex24x6.npy')
vertvis.plot_correlation(cor_mat, "Vertex Correlation")

## compare autoencoder models

In [ ]:
verteval.plot_all_rmses(rmse_df, figsize=(6, 6))

In [ ]:
grouped_df = rmse_df.groupby(['run_id', 'ld', 's'])
for (run_id, ld, s), group in grouped_df:
    verteval.print_rmses(group.to_dict(), f'{run_id}_ld{ld}_s{s}')

## compare phase classifier models

In [ ]:
verteval.plot_classification_results(classification_df)

In [ ]:
labels = list(AutoEncoderVertexDataset.phase_borders.keys())
for i, row in classification_df.iterrows():
    verteval.print_conf_mat(row['conf_mat'], f"{row['run_id']}_ld{row['ld']}_s{row['s']}", labels)

## compare vertex reconstructions